# NBA Data Exploratory Analysis

This notebook explores the NBA dataset used in the COSC480 project. We'll conduct a thorough analysis of the data to uncover trends and visualize key relationships between different statistics across teams and seasons.

## Contents
1. Data Loading and Inspection
2. Data Cleaning and Preparation
3. Exploratory Data Analysis
4. Statistical Analysis
5. Advanced Visualizations
6. Insights and Findings

## 1. Data Loading and Inspection

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-whitegrid')
sns.set_palette("deep")
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12

In [ ]:
# Load the data
def load_nba_data(file_path="ProjectData.txt"):
    with open(file_path, 'r') as file:
        lines = file.read().splitlines()
    
    # Skip the header lines
    data_lines = lines[4:]  # Assuming first 4 lines are headers
    
    # Extract team data
    mavericks_data = data_lines[1:4]  # Lines for Mavericks
    lakers_data = data_lines[4:7]     # Lines for Lakers
    knicks_data = data_lines[7:10]    # Lines for Knicks
    
    return {
        'mavericks': mavericks_data,
        'lakers': lakers_data,
        'knicks': knicks_data,
        'raw_lines': data_lines,
        'headers': lines[:4]
    }

# Load the data
nba_data = load_nba_data()
print("Data loaded successfully.")
print("Headers:")
for header in nba_data['headers']:
    print(header)

In [ ]:
# Create a structured dataframe from the raw data
def process_team_data(team_data_lines):
    processed_data = []
    
    for line in team_data_lines:
        parts = line.split()
        
        # Extract team and year
        team_info = parts[0:2]  # First two elements are team name and year
        team = team_info[0]
        year = team_info[1].strip('()')
        
        # Extract numerical data
        numerical_data = parts[2:]
        
        # Create a record
        record = {
            'team': team,
            'year': year,
            'win_pct_total': float(numerical_data[0]),
            'win_pct_regular': float(numerical_data[1]),
            'win_pct_playoff': float(numerical_data[2]),
            'three_pct_total': float(numerical_data[3]),
            'three_pct_regular': float(numerical_data[4]),
            'three_pct_playoff': float(numerical_data[5]),
            'points_total': float(numerical_data[6]),
            'points_regular': float(numerical_data[7]),
            'points_playoff': float(numerical_data[8]),
            'three_attempts_total': float(numerical_data[9]),
            'three_attempts_regular': float(numerical_data[10]),
            'three_attempts_playoff': float(numerical_data[11]),
            'wins_total': int(numerical_data[12]),
            'wins_regular': int(numerical_data[13]),
            'wins_playoff': int(numerical_data[14]),
            'losses_total': int(numerical_data[15]),
            'losses_regular': int(numerical_data[16]),
            'losses_playoff': int(numerical_data[17]),
            'three_makes_total': float(numerical_data[18]),
            'three_makes_regular': float(numerical_data[19]),
            'three_makes_playoff': float(numerical_data[20])
        }
        processed_data.append(record)
    
    return processed_data

# Process all team data
mavericks_records = process_team_data(nba_data['mavericks'])
lakers_records = process_team_data(nba_data['lakers'])
knicks_records = process_team_data(nba_data['knicks'])

# Combine all records into a single dataframe
all_records = mavericks_records + lakers_records + knicks_records
nba_df = pd.DataFrame(all_records)

# Display the dataframe
nba_df.head()

## 2. Data Cleaning and Preparation

In [ ]:
# Check for missing values
print("Missing values in each column:")
print(nba_df.isnull().sum())

# Check data types
print("\nData types:")
print(nba_df.dtypes)

# Basic statistics
print("\nBasic statistics:")
nba_df.describe()

In [ ]:
# Convert year to a numeric type
nba_df['year'] = nba_df['year'].astype(int)

# Create additional derived metrics
nba_df['games_played_total'] = nba_df['wins_total'] + nba_df['losses_total']
nba_df['games_played_regular'] = nba_df['wins_regular'] + nba_df['losses_regular']
nba_df['games_played_playoff'] = nba_df['wins_playoff'] + nba_df['losses_playoff']

# Calculate efficiency metrics
nba_df['points_per_game_total'] = nba_df['points_total']
nba_df['three_efficiency_total'] = nba_df['three_makes_total'] / nba_df['three_attempts_total'] * 100

# Display the updated dataframe
nba_df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Visualize win percentage by team and year
plt.figure(figsize=(14, 8))
sns.barplot(x='team', y='win_pct_total', hue='year', data=nba_df)
plt.title('Win Percentage by Team and Year', fontsize=16)
plt.xlabel('Team', fontsize=14)
plt.ylabel('Win Percentage', fontsize=14)
plt.ylim(0, 100)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title='Year', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Visualize 3-point percentage trends over time
plt.figure(figsize=(14, 8))
for team in nba_df['team'].unique():
    team_data = nba_df[nba_df['team'] == team]
    plt.plot(team_data['year'], team_data['three_pct_total'], marker='o', linewidth=3, markersize=10, label=team)

plt.title('3-Point Percentage Trends Over Time', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('3-Point Percentage', fontsize=14)
plt.xticks(nba_df['year'].unique(), fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Visualize 3-point attempts trends over time
plt.figure(figsize=(14, 8))
for team in nba_df['team'].unique():
    team_data = nba_df[nba_df['team'] == team]
    plt.plot(team_data['year'], team_data['three_attempts_total'], marker='o', linewidth=3, markersize=10, label=team)

plt.title('3-Point Attempts Trends Over Time', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('3-Point Attempts per Game', fontsize=14)
plt.xticks(nba_df['year'].unique(), fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 4. Statistical Analysis

In [ ]:
# Correlation analysis
correlation_columns = ['win_pct_total', 'three_pct_total', 'three_attempts_total', 'points_total']
correlation_matrix = nba_df[correlation_columns].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Between Key Metrics', fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()

print("Correlation Analysis:")
for col1 in correlation_columns:
    for col2 in correlation_columns:
        if col1 != col2:
            corr = nba_df[col1].corr(nba_df[col2])
            print(f"Correlation between {col1} and {col2}: {corr:.4f}")

In [ ]:
# Statistical comparison across years
year_stats = nba_df.groupby('year').agg({
    'win_pct_total': 'mean',
    'three_pct_total': 'mean',
    'three_attempts_total': 'mean',
    'points_total': 'mean',
    'three_makes_total': 'mean'
}).reset_index()

print("Average Statistics by Year:")
year_stats

In [ ]:
# Statistical comparison across teams
team_stats = nba_df.groupby('team').agg({
    'win_pct_total': 'mean',
    'three_pct_total': 'mean',
    'three_attempts_total': 'mean',
    'points_total': 'mean',
    'three_makes_total': 'mean'
}).reset_index()

print("Average Statistics by Team:")
team_stats

## 5. Advanced Visualizations

In [ ]:
# Scatter plot matrix
sns.pairplot(nba_df[correlation_columns + ['team']], hue='team', height=3, aspect=1.2)
plt.suptitle('Scatter Plot Matrix of Key Metrics', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 3D Scatter plot
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

colors = {'Mavericks': 'blue', 'Lakers': 'purple', 'Knicks': 'orange'}
markers = {2000: 'o', 2010: 's', 2020: '^'}

for team in nba_df['team'].unique():
    for year in nba_df['year'].unique():
        data = nba_df[(nba_df['team'] == team) & (nba_df['year'] == year)]
        ax.scatter(data['three_pct_total'], 
                  data['three_attempts_total'], 
                  data['win_pct_total'],
                  color=colors.get(team, 'black'),
                  marker=markers.get(year, 'o'),
                  s=100,
                  label=f"{team} {year}")

ax.set_xlabel('3-Point Percentage', fontsize=12)
ax.set_ylabel('3-Point Attempts per Game', fontsize=12)
ax.set_zlabel('Win Percentage', fontsize=12)
ax.set_title('3D Relationship Between 3-Point Metrics and Win Percentage', fontsize=16)

# Add a custom legend
team_legend = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=team) 
               for team, color in colors.items()]
year_legend = [plt.Line2D([0], [0], marker=marker, color='black', markersize=10, label=str(year)) 
               for year, marker in markers.items()]

ax.legend(handles=team_legend + year_legend, loc='best', title="Teams and Years")

plt.tight_layout()
plt.show()

In [ ]:
# Time series of 3-point metrics and win percentage
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 18), sharex=True)

for team in nba_df['team'].unique():
    team_data = nba_df[nba_df['team'] == team].sort_values('year')
    
    # Win percentage plot
    ax1.plot(team_data['year'], team_data['win_pct_total'], marker='o', linewidth=3, markersize=10, label=team)
    ax1.set_ylabel('Win Percentage', fontsize=14)
    ax1.set_title('Win Percentage Over Time', fontsize=16)
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.legend(fontsize=12)
    
    # 3-point percentage plot
    ax2.plot(team_data['year'], team_data['three_pct_total'], marker='o', linewidth=3, markersize=10, label=team)
    ax2.set_ylabel('3-Point Percentage', fontsize=14)
    ax2.set_title('3-Point Percentage Over Time', fontsize=16)
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    # 3-point attempts plot
    ax3.plot(team_data['year'], team_data['three_attempts_total'], marker='o', linewidth=3, markersize=10, label=team)
    ax3.set_ylabel('3-Point Attempts per Game', fontsize=14)
    ax3.set_title('3-Point Attempts Over Time', fontsize=16)
    ax3.grid(True, linestyle='--', alpha=0.7)

ax3.set_xlabel('Year', fontsize=14)
ax3.set_xticks(nba_df['year'].unique())
plt.tight_layout()
plt.show()

## 6. Insights and Findings

### Key Observations

1. **Evolution of 3-Point Shooting**: There has been a significant increase in 3-point attempts across all teams from 2000 to 2020, reflecting the league-wide shift towards perimeter shooting.

2. **Shooting Efficiency vs. Volume**: While 3-point attempts have increased dramatically, 3-point shooting percentages have remained relatively stable across the years, suggesting teams have managed to maintain efficiency despite higher volume.

3. **Win Percentage Correlation**: There appears to be a moderate positive correlation between 3-point shooting percentage and win percentage, but the relationship with 3-point attempt volume is more complex.

4. **Team Differences**: The data shows distinct differences in how each team has adapted to the evolution of the game:
   - Mavericks have shown the most consistent growth in 3-point volume while maintaining efficiency
   - Lakers have had more variable performance across metrics
   - Knicks show improvement in shooting volume but less consistency in outcomes

5. **Scoring Trends**: Overall scoring (points per game) has increased from 2000 to 2020, corresponding with the increase in 3-point attempts, demonstrating the impact of perimeter shooting on offensive production.

### Practical Applications

This analysis demonstrates how data visualization can reveal important trends in basketball performance metrics. These insights could be valuable for:

- Team management making strategic decisions about player acquisition and development
- Coaches developing offensive and defensive systems
- Analysts predicting team performance based on shooting metrics
- Fans gaining deeper understanding of how the game has evolved

The NBA data visualization tool developed for this project provides an accessible way to explore these relationships and generate similar insights across different combinations of teams, seasons, and statistics.

## Conclusion

This exploratory analysis has demonstrated the power of data visualization in uncovering patterns and relationships within NBA statistics. The original project tool provides an interactive way to explore these relationships, while this notebook extends the analysis with more sophisticated visualizations and statistical insights.

The evolution of NBA basketball towards a perimeter-oriented game is clearly visible in the data, with significant increases in 3-point attempt volume across all teams from 2000 to 2020. The relationship between shooting metrics and team success (win percentage) provides valuable insights into how the game has changed and which strategies have proven most effective.

For further exploration, additional data covering more teams and seasons would allow for more comprehensive analysis of league-wide trends and team-specific strategies.